# Lab 5b: Information Diffusion & Influence

**Course:** IS353 - Social Network Analysis, UIT, VNUHCM  
**Lecturer:** Dr. Tran Hung Nghiep, 2025  

---

**Goals:**
- Implement and simulate Independent Cascade (IC) and Linear Threshold (LT).
- Visualize cascades over time and plot adoption curves.
- Compare seed selection strategies (Random, Degree, PageRank).
- *(Bonus)* Sensor placement for outbreak detection.
- *(Optional Advanced)* Simple learning-based diffusion predictor.


## 0. Setup

In [ ]:
import math
import random
from collections import deque, defaultdict

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# plotting policy: one chart per figure, default styles only
plt.rcParams['figure.figsize'] = (7,5)
plt.rcParams['axes.grid'] = True

random.seed(42)
np.random.seed(42)

## 1. Graphs for the Lab

In [ ]:
# Use Zachary's Karate Club for fast iteration
G = nx.karate_club_graph()

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

# Precompute a fixed layout for consistent drawings
pos = nx.spring_layout(G, seed=7)

# Quick look
plt.figure()
nx.draw(G, pos, with_labels=True)
plt.title("Zachary Karate Club Graph")
plt.show()

## 2. Independent Cascade (IC)

**Model recap**  
At each discrete step, every newly-activated node gets **one chance** to activate each inactive neighbor with probability $p$. Activated nodes remain active.

In [ ]:
def independent_cascade(G, seeds, p=0.1, steps=50):
    active = set(seeds)
    newly_active = set(seeds)
    history = [set(active)]  # list of sets of active nodes after each step

    for _ in range(steps):
        next_active = set()
        for u in newly_active:
            for v in G.neighbors(u):
                if v in active:
                    continue
                if random.random() < p:
                    next_active.add(v)
        if not next_active:
            break
        active |= next_active
        newly_active = next_active
        history.append(set(active))
    return history

# Demo
seeds = [0]
hist_ic = independent_cascade(G, seeds, p=0.15)
print("IC steps:", len(hist_ic), "| final active:", len(hist_ic[-1]))

# Adoption curve
plt.figure()
plt.plot([len(s) for s in hist_ic], marker='o')
plt.xlabel("Step")
plt.ylabel("# Active nodes")
plt.title("IC: Adoption Curve")
plt.show()

## 3. Linear Threshold (LT)

**Model recap**  
Assign each node a threshold $\theta_v\in[0,1]$. A node activates when the **fraction of active neighbors** exceeds its threshold (simple unweighted LT here). Activation is monotone.

In [ ]:
def linear_threshold(G, seeds, thresholds=None, steps=50):
    if thresholds is None:
        thresholds = {v: random.random() for v in G.nodes()}
    active = set(seeds)
    history = [set(active)]
    for _ in range(steps):
        new_active = set()
        for v in G.nodes():
            if v in active:
                continue
            nbrs = list(G.neighbors(v))
            if not nbrs:
                continue
            frac = len(set(nbrs) & active) / len(nbrs)
            if frac >= thresholds[v]:
                new_active.add(v)
        if not new_active:
            break
        active |= new_active
        history.append(set(active))
    return history, thresholds

# Demo
seeds = [0]
hist_lt, th = linear_threshold(G, seeds)
print("LT steps:", len(hist_lt), "| final active:", len(hist_lt[-1]))

plt.figure()
plt.plot([len(s) for s in hist_lt], marker='o')
plt.xlabel("Step")
plt.ylabel("# Active nodes")
plt.title("LT: Adoption Curve")
plt.show()

## 4. Visualizing Cascades (Node Size = Active)

For clarity and to avoid color coding, we enlarge **active** nodes at each step.  
Use this sparingly (first ~5 steps) to keep the notebook light.

In [ ]:
def draw_step(G, active_set, step, pos):
    sizes = [300 + (700 if n in active_set else 0) for n in G.nodes()]
    plt.figure()
    nx.draw(G, pos, with_labels=True, node_size=sizes)
    plt.title(f"Cascade Step {step}")
    plt.show()

# Show first few steps for IC
for t, act in enumerate(hist_ic[:5]):
    draw_step(G, act, t, pos)

## 5. Influence Maximization with Heuristic Seeding

Compare three strategies for choosing **k** seeds:  
- **Random** nodes  
- **Top-Degree** nodes  
- **Top-PageRank** nodes  
Evaluate by final cascade size under IC.

In [ ]:
def choose_seeds(G, k, strategy="random"):
    nodes = list(G.nodes())
    if strategy == "random":
        return random.sample(nodes, k)
    if strategy == "degree":
        return [n for n,_ in sorted(G.degree, key=lambda x: x[1], reverse=True)[:k]]
    if strategy == "pagerank":
        pr = nx.pagerank(G)
        return [n for n,_ in sorted(pr.items(), key=lambda x: x[1], reverse=True)[:k]]
    raise ValueError("Unknown strategy")

def evaluate_strategies(G, k=3, p=0.15, trials=20):
    strategies = ["random","degree","pagerank"]
    res = {s: [] for s in strategies}
    for s in strategies:
        for _ in range(trials):
            seeds = choose_seeds(G, k, s)
            hist = independent_cascade(G, seeds, p=p)
            res[s].append(len(hist[-1]))
    return {s: (float(np.mean(v)), float(np.std(v))) for s,v in res.items()}

res = evaluate_strategies(G, k=3, p=0.15, trials=50)
print("Average final spread (± std):")
for k,(m,sd) in res.items():
    print(f"{k:9s}: {m:.2f} ± {sd:.2f}")

plt.figure()
labels = list(res.keys())
means = [res[k][0] for k in labels]
plt.bar(labels, means)
plt.ylabel("Avg final #active")
plt.title("Seeding Strategy Comparison (IC)")
plt.show()

## 6. Bonus: Outbreak Detection (Sensor Placement)

**Goal**: pick $k$ **sensors** to **detect** an outbreak early (small first-hit time).  
Heuristic: choose nodes minimizing average distance to others (closeness) as sensors.

In [ ]:
def simulate_detection_time(G, seeds, sensors, p=0.15, trials=50):
    # Run many IC cascades; record the first time any sensor becomes active.
    times = []
    for _ in range(trials):
        hist = independent_cascade(G, seeds, p=p, steps=100)
        first_time = None
        for t, active in enumerate(hist):
            if any(s in active for s in sensors):
                first_time = t
                break
        times.append(first_time if first_time is not None else float('inf'))
    finite = [t for t in times if math.isfinite(t)]
    return (np.mean(finite) if finite else float('inf'), len(finite)/trials)

# Choose sensors by closeness centrality
k = 3
cl = nx.closeness_centrality(G)
sensors = [n for n,_ in sorted(cl.items(), key=lambda x: x[1], reverse=True)[:k]]
seeds = choose_seeds(G, 3, "random")

mean_time, detect_rate = simulate_detection_time(G, seeds, sensors, p=0.15, trials=100)
print("Sensors:", sensors)
print("Mean detection time:", round(mean_time,2), "| detection rate:", round(detect_rate,2))

## 7. (Optional) Advanced: Simple Learning-Based Diffusion Predictor

Create a small labeled dataset of node pairs observed to transmit in IC simulations (positive) vs. not (negative).  
Features: degree(u), degree(v), PageRank(u), PageRank(v), shortest-path length between u and v.  
Train a logistic regression classifier and evaluate with accuracy/AP.

> This section is optional and may require `scikit-learn`.

In [ ]:
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import average_precision_score, accuracy_score
    from sklearn.model_selection import train_test_split
    SK_OK = True
except Exception:
    SK_OK = False

def sample_ic_pairs(G, p=0.15, trials=100):
    # Collect (u->v) pairs that became active at t when u was newly active at t-1
    pos_pairs = set()
    for _ in range(trials):
        seeds = [random.choice(list(G.nodes()))]
        hist = independent_cascade(G, seeds, p=p, steps=20)
        newly = [hist[0]]
        for t in range(1, len(hist)):
            newly.append(hist[t] - hist[t-1])
        for t in range(1, len(newly)):
            for u in newly[t-1]:
                for v in newly[t]:
                    if G.has_edge(u,v) or G.has_edge(v,u):
                        pos_pairs.add((u,v))
    return list(pos_pairs)

def make_features(G, pairs):
    pr = nx.pagerank(G)
    X = []
    for u,v in pairs:
        sp = np.inf
        try:
            sp = nx.shortest_path_length(G, u, v)
        except nx.NetworkXNoPath:
            pass
        X.append([G.degree(u), G.degree(v), pr[u], pr[v], float(sp) if sp!=np.inf else 1e9])
    return np.array(X)

if SK_OK:
    pos = sample_ic_pairs(G, p=0.15, trials=60)
    all_nodes = list(G.nodes())
    neg_candidates = []
    for _ in range(len(pos)*2 + 50):
        u = random.choice(all_nodes)
        v = random.choice(all_nodes)
        if u==v:
            continue
        if (u,v) in pos or (v,u) in pos:
            continue
        if not G.has_edge(u,v) and not G.has_edge(v,u):
            neg_candidates.append((u,v))
    neg = neg_candidates[:len(pos)]

    pairs = pos + neg
    y = np.array([1]*len(pos) + [0]*len(neg))
    X = make_features(G, pairs)
    # Replace inf path with large finite
    X[:,4] = np.where(np.isinf(X[:,4]), 1e9, X[:,4])

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    yp = clf.predict(Xte)
    proba = clf.predict_proba(Xte)[:,1]
    print("Learning-based predictor. Accuracy:", round(accuracy_score(yte, yp),3),
          " AP:", round(average_precision_score(yte, proba),3))
else:
    print("scikit-learn not available -> skip optional learning section.")

## 8. Questions

1. Completed notebook with outputs:
    - IC and LT implementations and adoption curves.
    - Cascade visualizations (first few steps).
    - Influence maximization comparison bar chart.
2. What structural factors most affect cascade size and speed in your experiments?
3. *(Bonus)* Outbreak detection results and a brief interpretation.
4. *(Optional)* Learning-based predictor metrics and comments.
5. *(Optional)* Answer the questions using the larger ego-Facebook dataset.